# 2.1 — Data Preparation (Manual Pipeline)

This notebook implements the full data preparation pipeline for the **Hip Replacement** dataset, following the CRISP-DM methodology.

## Pipeline Overview

| Step | Operation | Applied Before / After Split |
|---|---|---|
| 1 | Verify indicator columns | Before |
| 2 | Listwise deletion — OHS pre-op 12 dimensions | Before |
| 3 | Calculate derived scores (Pre-Op Q Score, Post-Op Q Score, EQ5D profiles & indices) | Before |
| 4 | Listwise deletion — EQ-5D dimensions | Before |
| 5 | Listwise deletion — Symptom Period | Before |
| 6 | Remove irrelevant columns (Post-Op questions, Predicted, CSVYear) | Before |
| 7 | Listwise deletion — Age Band & Gender | Before |
| 8 | Listwise deletion — Post-Op Q Score (non-responders) & drop EQ VAS columns | Before |
| 9 | Train / Test split (80 / 20, seed=42) | — |
| 10 | Mode imputation & constant fills | **After** |
| 11 | Create binary outcome variable `OHS_Success` | **After** |
| 12 | Save train & test datasets | **After** |

In [ ]:
# Import Required Libraries
import polars as pl
import numpy as np

In [139]:
pl.Config.set_tbl_cols(-1)       # display.max.columns -> None (-1 means all)
pl.Config.set_tbl_rows(-1)       # display.max.rows -> None (-1 means all)
pl.Config.set_float_precision(2) # display.precision -> 2

polars.config.Config

## Setup — Load Data & Explore

In [ ]:
# Load the Hip Replacement CCG data
# Load the Knee Replacement Provider data
df = pl.read_parquet(f'./data/interim/2.0-preprocessing.parquet')


# Make a copy of the original dataframe
df_original = df.clone()
print("Original dataframe copied to df_original.")

Original dataframe copied to df_original.


In [141]:
# Display basic information about the dataset
print("Shape of the dataset:", df.shape)
print("\nColumn names:")
print(df.columns)
print("\nData types:")
print(df.schema)
print("\nFirst 5 rows:")
df.head()

Shape of the dataset: (124844, 82)

Column names:
['Provider Code', 'Procedure', 'Revision Flag', 'Year', 'Age Band', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis', 'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety', 'Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Post-Op Q Assisted', 'Post-Op Q Assisted By', 'Post-Op Q Living Arrangements', 'Post-Op Q Disability', 'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety', 'Post-Op Q Satisfaction', 'Post-Op Q Sucess', 'Post-Op Q Allergy', 'Post-Op Q Bleeding', 'Post-Op Q Wound', 'Post-Op Q Urine', 'Post-Op Q Further Surgery', 'Post-Op Q R

Provider Code,Procedure,Revision Flag,Year,Age Band,Gender,Pre-Op Q Assisted,Pre-Op Q Assisted By,Pre-Op Q Symptom Period,Pre-Op Q Previous Surgery,Pre-Op Q Living Arrangements,Pre-Op Q Disability,Heart Disease,High Bp,Stroke,Circulation,Lung Disease,Diabetes,Kidney Disease,Nervous System,Liver Disease,Cancer,Depression,Arthritis,Pre-Op Q Mobility,Pre-Op Q Self-Care,Pre-Op Q Activity,Pre-Op Q Discomfort,Pre-Op Q Anxiety,Pre-Op Q EQ5D Index Profile,Pre-Op Q EQ5D Index,Post-Op Q Assisted,Post-Op Q Assisted By,Post-Op Q Living Arrangements,Post-Op Q Disability,Post-Op Q Mobility,Post-Op Q Self-Care,Post-Op Q Activity,Post-Op Q Discomfort,Post-Op Q Anxiety,Post-Op Q Satisfaction,Post-Op Q Sucess,Post-Op Q Allergy,Post-Op Q Bleeding,Post-Op Q Wound,Post-Op Q Urine,Post-Op Q Further Surgery,Post-Op Q Readmitted,Post-Op Q EQ5D Index Profile,Post-Op Q EQ5D Index,Hip Replacement EQ5D Index Post-Op Q Predicted,Pre-Op Q EQ VAS,Post-Op Q EQ VAS,Hip Replacement EQ VAS Post-Op Q Predicted,Hip Replacement Pre-Op Q Pain,Hip Replacement Pre-Op Q Sudden Pain,Hip Replacement Pre-Op Q Night Pain,Hip Replacement Pre-Op Q Washing,Hip Replacement Pre-Op Q Transport,Hip Replacement Pre-Op Q Dressing,Hip Replacement Pre-Op Q Shopping,Hip Replacement Pre-Op Q Walking,Hip Replacement Pre-Op Q Limping,Hip Replacement Pre-Op Q Stairs,Hip Replacement Pre-Op Q Standing,Hip Replacement Pre-Op Q Work,Hip Replacement Pre-Op Q Score,Hip Replacement Post-Op Q Pain,Hip Replacement Post-Op Q Sudden Pain,Hip Replacement Post-Op Q Night Pain,Hip Replacement Post-Op Q Washing,Hip Replacement Post-Op Q Transport,Hip Replacement Post-Op Q Dressing,Hip Replacement Post-Op Q Shopping,Hip Replacement Post-Op Q Walking,Hip Replacement Post-Op Q Limping,Hip Replacement Post-Op Q Stairs,Hip Replacement Post-Op Q Standing,Hip Replacement Post-Op Q Work,Hip Replacement Post-Op Q Score,Hip Replacement OHS Post-Op Q Predicted,CSVYear
cat,cat,u8,cat,cat,f32,u8,i32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u32,f32,u8,i32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u32,f32,f32,u16,u16,cat,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,f32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,f32,cat,cat
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,2,1,1,1,0,0,0,0,0,0,1,0,0,0,0,1,null,null,null,null,null,99999,null,2,0,2,2,1,1,2,1,1,2,1,2,2,2,2,2,2,11211,0.88,null,null,80,null,0,2,1,3,1,2,3,2,1,0,2,2,19.00,3,4,2,4,3,2,4,4,3,2,2,3,36.00,"""3.777.303.196""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,3,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,3,1,22231,0.05,2,0,1,2,2,1,1,1,1,1,1,2,2,2,2,2,2,21111,0.85,0.67,30,70,"""6.592.244.892""",0,1,0,2,2,1,1,1,0,2,2,1,13.00,4,4,4,4,3,3,4,4,4,4,4,4,46.00,"""3.558.681.573""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,1,1,2,2,1,1,0,1,0,0,0,0,0,0,0,0,0,1,2,2,3,3,2,22332,-0.07,2,0,1,1,2,1,2,1,1,2,1,2,2,2,2,2,2,21211,0.81,0.70,40,60,"""6.810.332.786""",1,0,0,1,1,1,0,1,0,1,1,0,7.00,1,4,3,3,2,3,2,4,1,2,4,2,31.00,"""3.293.405.433""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,4,2,1,2,0,0,0,0,0,0,0,1,0,0,0,0,2,2,2,3,1,22231,0.05,2,0,1,2,2,1,2,2,2,3,2,2,2,2,1,2,1,21222,0.62,0.71,50,55,"""6.883.315.758""",0,4,0,2,2,1,3,2,1,2,2,2,21.00,0,0,0,3,2,2,2,1,1,2,2,0,15.00,"""3.867.962.866""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,4,2,1,2,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,3,2,22232,-0.02,2,0,1,2,1,1,1,1,1,1,1,2,2,2,2,2,2,11111,1.00,0.74,20,90,"""7.226.409.643""",0,0,0,1,1,1,2,2,0,2,1,0,10.00,4,4,3,4,4,4,4,4,4,4,4,4,47.00,"""3.686.746.026""","""2016/17"""


In [142]:
# Show only columns that have at least one null value
null_counts = df.null_count()
cols_with_nulls = [c for c in df.columns if null_counts[c][0] > 0]
print(f"Columns with null values ({len(cols_with_nulls)}):")
print(null_counts.select(cols_with_nulls))

Columns with null values (61):
shape: (1, 61)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ Age ┆ Gen ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Hip ┆ Pre ┆ Pos ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip │
│ Ban ┆ der ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ t-O ┆ Rep ┆ -Op ┆ t-O ┆ Rep ┆ Rep ┆ Rep ┆ 

In [143]:
# Identify column types
numeric_dtypes = [pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, pl.Float32, pl.Float64]
string_dtypes  = [pl.Utf8, pl.Categorical]  # parquet may store string columns as Categorical

numerical_cols = [c for c in df.columns if df[c].dtype in numeric_dtypes]
string_cols    = [c for c in df.columns if df[c].dtype in string_dtypes]

numerical_df = df.select(numerical_cols)
string_df    = df.select(string_cols)

print("List of numerical columns:")
print(numerical_cols)
print(f"\nList of string/categorical columns:")
print(string_cols)
print(f"\nNumerical dataframe shape: {numerical_df.shape}")
print(df.describe())

List of numerical columns:
['Revision Flag', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis', 'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety', 'Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Post-Op Q Assisted', 'Post-Op Q Assisted By', 'Post-Op Q Living Arrangements', 'Post-Op Q Disability', 'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety', 'Post-Op Q Satisfaction', 'Post-Op Q Sucess', 'Post-Op Q Allergy', 'Post-Op Q Bleeding', 'Post-Op Q Wound', 'Post-Op Q Urine', 'Post-Op Q Further Surgery', 'Post-Op Q Readmitted', 'Post-Op Q EQ5D Index Profile', 'Post-Op Q EQ5D Index', 'Hip 

In [144]:
# Print dataset overview
print("DATASET OVERVIEW:")
print(f"Total observations: {df.shape[0]}")
print(f"Total variables/columns: {df.shape[1]}")
print(f"\nNumerical variables/columns: {len(numerical_cols)}")
print(f"String variables/columns: {len(string_cols)}")

print(f"\nNumerical columns ({len(numerical_cols)}):")
print(numerical_cols)

print(f"\nString columns ({len(string_cols)}):")
print(string_cols)

print(f"\nNumerical dataframe shape: {numerical_df.shape}")
print(f"String dataframe shape: {string_df.shape}")

DATASET OVERVIEW:
Total observations: 124844
Total variables/columns: 82

Numerical variables/columns: 75
String variables/columns: 7

Numerical columns (75):
['Revision Flag', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis', 'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety', 'Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Post-Op Q Assisted', 'Post-Op Q Assisted By', 'Post-Op Q Living Arrangements', 'Post-Op Q Disability', 'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety', 'Post-Op Q Satisfaction', 'Post-Op Q Sucess', 'Post-Op Q Allergy', 'Post-Op Q Bleeding', 'Post-Op Q Wound', '

In [145]:
# Check for missing values
null_counts = df.null_count()
print("Missing values per column:")
print(null_counts)

print("\nPercentage of missing values:")
print(null_counts / df.shape[0] * 100)

print("\nFirst 10 rows:")
df.head(10)

Missing values per column:
shape: (1, 82)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ Pro ┆ Pro ┆ Rev ┆ Yea ┆ Age ┆ Gen ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Hea ┆ Hig ┆ Str ┆ Cir ┆ Lun ┆ Dia ┆ Kid ┆ Ner ┆ Liv ┆ Can ┆ Dep ┆ Art ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Hip ┆ Pre ┆ Pos ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ 

Provider Code,Procedure,Revision Flag,Year,Age Band,Gender,Pre-Op Q Assisted,Pre-Op Q Assisted By,Pre-Op Q Symptom Period,Pre-Op Q Previous Surgery,Pre-Op Q Living Arrangements,Pre-Op Q Disability,Heart Disease,High Bp,Stroke,Circulation,Lung Disease,Diabetes,Kidney Disease,Nervous System,Liver Disease,Cancer,Depression,Arthritis,Pre-Op Q Mobility,Pre-Op Q Self-Care,Pre-Op Q Activity,Pre-Op Q Discomfort,Pre-Op Q Anxiety,Pre-Op Q EQ5D Index Profile,Pre-Op Q EQ5D Index,Post-Op Q Assisted,Post-Op Q Assisted By,Post-Op Q Living Arrangements,Post-Op Q Disability,Post-Op Q Mobility,Post-Op Q Self-Care,Post-Op Q Activity,Post-Op Q Discomfort,Post-Op Q Anxiety,Post-Op Q Satisfaction,Post-Op Q Sucess,Post-Op Q Allergy,Post-Op Q Bleeding,Post-Op Q Wound,Post-Op Q Urine,Post-Op Q Further Surgery,Post-Op Q Readmitted,Post-Op Q EQ5D Index Profile,Post-Op Q EQ5D Index,Hip Replacement EQ5D Index Post-Op Q Predicted,Pre-Op Q EQ VAS,Post-Op Q EQ VAS,Hip Replacement EQ VAS Post-Op Q Predicted,Hip Replacement Pre-Op Q Pain,Hip Replacement Pre-Op Q Sudden Pain,Hip Replacement Pre-Op Q Night Pain,Hip Replacement Pre-Op Q Washing,Hip Replacement Pre-Op Q Transport,Hip Replacement Pre-Op Q Dressing,Hip Replacement Pre-Op Q Shopping,Hip Replacement Pre-Op Q Walking,Hip Replacement Pre-Op Q Limping,Hip Replacement Pre-Op Q Stairs,Hip Replacement Pre-Op Q Standing,Hip Replacement Pre-Op Q Work,Hip Replacement Pre-Op Q Score,Hip Replacement Post-Op Q Pain,Hip Replacement Post-Op Q Sudden Pain,Hip Replacement Post-Op Q Night Pain,Hip Replacement Post-Op Q Washing,Hip Replacement Post-Op Q Transport,Hip Replacement Post-Op Q Dressing,Hip Replacement Post-Op Q Shopping,Hip Replacement Post-Op Q Walking,Hip Replacement Post-Op Q Limping,Hip Replacement Post-Op Q Stairs,Hip Replacement Post-Op Q Standing,Hip Replacement Post-Op Q Work,Hip Replacement Post-Op Q Score,Hip Replacement OHS Post-Op Q Predicted,CSVYear
cat,cat,u8,cat,cat,f32,u8,i32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u32,f32,u8,i32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u32,f32,f32,u16,u16,cat,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,f32,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,f32,cat,cat
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,2,1,1,1,0,0,0,0,0,0,1,0,0,0,0,1,null,null,null,null,null,99999,null,2,0,2,2,1,1,2,1,1,2,1,2,2,2,2,2,2,11211,0.88,null,null,80,null,0,2,1,3,1,2,3,2,1,0,2,2,19.00,3,4,2,4,3,2,4,4,3,2,2,3,36.00,"""3.777.303.196""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,3,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,3,1,22231,0.05,2,0,1,2,2,1,1,1,1,1,1,2,2,2,2,2,2,21111,0.85,0.67,30,70,"""6.592.244.892""",0,1,0,2,2,1,1,1,0,2,2,1,13.00,4,4,4,4,3,3,4,4,4,4,4,4,46.00,"""3.558.681.573""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,1,1,2,2,1,1,0,1,0,0,0,0,0,0,0,0,0,1,2,2,3,3,2,22332,-0.07,2,0,1,1,2,1,2,1,1,2,1,2,2,2,2,2,2,21211,0.81,0.70,40,60,"""6.810.332.786""",1,0,0,1,1,1,0,1,0,1,1,0,7.00,1,4,3,3,2,3,2,4,1,2,4,2,31.00,"""3.293.405.433""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,4,2,1,2,0,0,0,0,0,0,0,1,0,0,0,0,2,2,2,3,1,22231,0.05,2,0,1,2,2,1,2,2,2,3,2,2,2,2,1,2,1,21222,0.62,0.71,50,55,"""6.883.315.758""",0,4,0,2,2,1,3,2,1,2,2,2,21.00,0,0,0,3,2,2,2,1,1,2,2,0,15.00,"""3.867.962.866""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,4,2,1,2,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,3,2,22232,-0.02,2,0,1,2,1,1,1,1,1,1,1,2,2,2,2,2,2,11111,1.00,0.74,20,90,"""7.226.409.643""",0,0,0,1,1,1,2,2,0,2,1,0,10.00,4,4,3,4,4,4,4,4,4,4,4,4,47.00,"""3.686.746.026""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,2,2,1,1,0,1,0,0,0,0,0,0,0,0,0,1,2,1,2,3,2,21232,0.09,2,0,1,2,1,1,1,1,1,1,1,2,2,2,2,2,2,11111,1.00,0.75,80,100,"""7.907.610.235""",0,1,1,4,2,3,3,3,1,3,3,1,25.00,3,4,4,4,4,4,4,4,4,4,4,3,46.00,"""4.162.250.126""","""2016/17"""
"""ADP02""","""Hip Replacement""",0,"""2016/17""",null,null,2,1,1,2,1,1,0,1,0,0,0,0,0,0,0,0,0,

In [146]:
# Analyze missing values by Age Band
missing_by_age = (
    df.group_by('Age Band')
    .agg([pl.col(c).is_null().sum().alias(c) for c in df.columns if c != 'Age Band'])
    .sort('Age Band')
)
print("Missing values by Age Band:")
print(missing_by_age)

Missing values by Age Band:
shape: (9, 82)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ Age ┆ Pro ┆ Pro ┆ Rev ┆ Yea ┆ Gen ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Hea ┆ Hig ┆ Str ┆ Cir ┆ Lun ┆ Dia ┆ Kid ┆ Ner ┆ Liv ┆ Can ┆ Dep ┆ Art ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Pos ┆ Hip ┆ Pre ┆ Pos ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆

## Step 1 — Verify Indicator Columns

Check that binary comorbidity indicators have no residual missing values (value `9` was already replaced with `0` in the previous notebook).

In [147]:
# Replace missing values (originally 9) with 0 in binary indicator columns
# Already handlled by previous notebook, but we can verify that all 9s have been replaced
indicator_columns = [
    'Arthritis', 'Cancer', 'Circulation', 'Depression', 'Diabetes',
    'Heart Disease', 'High Bp', 'Kidney Disease', 'Liver Disease',
    'Lung Disease', 'Nervous System', 'Stroke'
]

# Check which indicator columns exist and have null values
existing_cols = [c for c in indicator_columns if c in df.columns]
missing_cols  = [c for c in indicator_columns if c not in df.columns]

if missing_cols:
    print(f"Columns NOT found in dataframe: {missing_cols}")

null_check = df.select(existing_cols).null_count()
print("Null counts for indicator columns:")
print(null_check)


Null counts for indicator columns:
shape: (1, 12)
┌────────┬────────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ Arthri ┆ Cancer ┆ Circul ┆ Depre ┆ Diabe ┆ Heart ┆ High  ┆ Kidne ┆ Liver ┆ Lung  ┆ Nervo ┆ Strok │
│ tis    ┆ ---    ┆ ation  ┆ ssion ┆ tes   ┆ Disea ┆ Bp    ┆ y Dis ┆ Disea ┆ Disea ┆ us    ┆ e     │
│ ---    ┆ u32    ┆ ---    ┆ ---   ┆ ---   ┆ se    ┆ ---   ┆ ease  ┆ se    ┆ se    ┆ Syste ┆ ---   │
│ u32    ┆        ┆ u32    ┆ u32   ┆ u32   ┆ ---   ┆ u32   ┆ ---   ┆ ---   ┆ ---   ┆ m     ┆ u32   │
│        ┆        ┆        ┆       ┆       ┆ u32   ┆       ┆ u32   ┆ u32   ┆ u32   ┆ ---   ┆       │
│        ┆        ┆        ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆ u32   ┆       │
╞════════╪════════╪════════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╡
│ 0      ┆ 0      ┆ 0      ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     ┆ 0     │
└────────┴────────┴────────┴───────┴─────

## Step 2 — Listwise Deletion: Oxford Hip Score Pre-Op Dimensions

Delete patient records where **any** of the 12 Oxford Hip Score (OHS) pre-operative dimension questions are unanswered.

These 12 columns are the primary disease-specific measure for hip replacements. Rows with missing values are removed because:
- The OHS dimension questions (e.g. `HR_Q1_LIMPING`) are among the **top predictors** for post-operative outcomes.
- Imputing values for such high-impact variables would introduce significant noise and bias.
- The research literature states: *"We removed observations with missing values"* to ensure classifiers are trained on fully informative data.

Missing rates for these columns range from 0.11 % to 0.94 %.

In [148]:
# Remove rows where any of the 12 Hip Replacement Pre-Op Q dimensions have missing values
hr_pre_cols = [col for col in df.columns if col.startswith('Hip Replacement Pre-Op Q') and col != 'Hip Replacement Pre-Op Q Score']

print("Hip Replacement Pre-Op Q columns (excluding Score):", hr_pre_cols)
print("Number of columns:", len(hr_pre_cols))

initial_shape = df.shape
df = df.drop_nulls(subset=hr_pre_cols)
final_shape = df.shape

print(f"Removed {initial_shape[0] - final_shape[0]} rows with missing values in Hip Replacement Pre-Op Q dimensions.")
print(f"New dataframe shape: {final_shape}")

Hip Replacement Pre-Op Q columns (excluding Score): ['Hip Replacement Pre-Op Q Pain', 'Hip Replacement Pre-Op Q Sudden Pain', 'Hip Replacement Pre-Op Q Night Pain', 'Hip Replacement Pre-Op Q Washing', 'Hip Replacement Pre-Op Q Transport', 'Hip Replacement Pre-Op Q Dressing', 'Hip Replacement Pre-Op Q Shopping', 'Hip Replacement Pre-Op Q Walking', 'Hip Replacement Pre-Op Q Limping', 'Hip Replacement Pre-Op Q Stairs', 'Hip Replacement Pre-Op Q Standing', 'Hip Replacement Pre-Op Q Work']
Number of columns: 12
Removed 1417 rows with missing values in Hip Replacement Pre-Op Q dimensions.
New dataframe shape: (123427, 82)


## Step 3 — Calculate Derived Scores from Dimensions

Missing values for `Hip Replacement Pre-Op Q Score` can be calculated by **summing** the 12 pre-op dimension columns (Oxford Hip Score = sum of 12 questions).

Missing values for `Hip Replacement Post-Op Q Score` can likewise be calculated by summing the 12 post-op dimension columns.

Missing values for `Pre-Op Q EQ5D Index Profile` can be calculated by **concatenating** `Pre-Op Q Mobility`, `Pre-Op Q Self-Care`, `Pre-Op Q Activity`, `Pre-Op Q Discomfort`, `Pre-Op Q Anxiety` as a 5-digit string.

Missing values for `Post-Op Q EQ5D Index Profile` can similarly be calculated by concatenating the 5 post-op EQ-5D columns.

> These calculations are performed **before the train/test split** and do not introduce leakage, as they are deterministic derivations from other observed columns in the same row.

In [149]:
# Calculate missing values for Hip Replacement Pre-Op Q Score by summing the 12 dimension columns
t0_hr_cols = [col for col in df.columns if col.startswith('Hip Replacement Pre-Op Q') and col != 'Hip Replacement Pre-Op Q Score']

print("Hip Replacement Pre-Op Q columns (excluding Score):", t0_hr_cols)
print("Number of Hip Replacement Pre-Op Q columns:", len(t0_hr_cols))

if len(t0_hr_cols) == 12:
    df = df.with_columns(
        pl.when(pl.col('Hip Replacement Pre-Op Q Score').is_null())
          .then(pl.sum_horizontal([pl.col(c) for c in t0_hr_cols]))
          .otherwise(pl.col('Hip Replacement Pre-Op Q Score'))
          .alias('Hip Replacement Pre-Op Q Score')
    )
    print("Missing values in Hip Replacement Pre-Op Q Score filled by summing the 12 Hip Replacement Pre-Op Q columns.")
    print(f"Remaining nulls in Pre-Op Q Score: {df['Hip Replacement Pre-Op Q Score'].null_count()}")
else:
    print(f"Warning: Expected 12 Hip Replacement Pre-Op Q columns, but found {len(t0_hr_cols)}.")

Hip Replacement Pre-Op Q columns (excluding Score): ['Hip Replacement Pre-Op Q Pain', 'Hip Replacement Pre-Op Q Sudden Pain', 'Hip Replacement Pre-Op Q Night Pain', 'Hip Replacement Pre-Op Q Washing', 'Hip Replacement Pre-Op Q Transport', 'Hip Replacement Pre-Op Q Dressing', 'Hip Replacement Pre-Op Q Shopping', 'Hip Replacement Pre-Op Q Walking', 'Hip Replacement Pre-Op Q Limping', 'Hip Replacement Pre-Op Q Stairs', 'Hip Replacement Pre-Op Q Standing', 'Hip Replacement Pre-Op Q Work']
Number of Hip Replacement Pre-Op Q columns: 12
Missing values in Hip Replacement Pre-Op Q Score filled by summing the 12 Hip Replacement Pre-Op Q columns.
Remaining nulls in Pre-Op Q Score: 0


In [150]:
# Calculate missing values for Hip Replacement Post-Op Q Score by summing the 12 dimension columns
t1_hr_cols = [col for col in df.columns if col.startswith('Hip Replacement Post-Op Q') and col != 'Hip Replacement Post-Op Q Score' and 'Predicted' not in col]

print("Hip Replacement Post-Op Q columns (excluding Score and Predicted):", t1_hr_cols)
print("Number of Hip Replacement Post-Op Q columns:", len(t1_hr_cols))

# Check for nulls in these columns before summing
null_check = df.select(t1_hr_cols).null_count()
print("\nNull counts in Post-Op Q dimension columns:")
print(null_check)

if len(t1_hr_cols) == 12:
    # True for any row where at least one component is null
    has_null = pl.any_horizontal([pl.col(c).is_null() for c in t1_hr_cols])

    df = df.with_columns(
        pl.when(has_null)
          # Any null component → score must be null (partial responses are invalid)
          .then(None)
          .when(pl.col('Hip Replacement Post-Op Q Score').is_null())
          # Score missing but all components present → calculate it
          .then(pl.sum_horizontal([pl.col(c) for c in t1_hr_cols]))
          .otherwise(pl.col('Hip Replacement Post-Op Q Score'))
          .alias('Hip Replacement Post-Op Q Score')
    )
    print("\nPost-Op Q Score: set to null where any component is null; filled by sum where score was missing.")
    print(f"Remaining nulls in Post-Op Q Score: {df['Hip Replacement Post-Op Q Score'].null_count()}")
else:
    print(f"Warning: Expected 12 Hip Replacement Post-Op Q columns, but found {len(t1_hr_cols)}.")

Hip Replacement Post-Op Q columns (excluding Score and Predicted): ['Hip Replacement Post-Op Q Pain', 'Hip Replacement Post-Op Q Sudden Pain', 'Hip Replacement Post-Op Q Night Pain', 'Hip Replacement Post-Op Q Washing', 'Hip Replacement Post-Op Q Transport', 'Hip Replacement Post-Op Q Dressing', 'Hip Replacement Post-Op Q Shopping', 'Hip Replacement Post-Op Q Walking', 'Hip Replacement Post-Op Q Limping', 'Hip Replacement Post-Op Q Stairs', 'Hip Replacement Post-Op Q Standing', 'Hip Replacement Post-Op Q Work']
Number of Hip Replacement Post-Op Q columns: 12

Null counts in Post-Op Q dimension columns:
shape: (1, 12)
┌────────┬────────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ Hip    ┆ Hip    ┆ Hip    ┆ Hip   ┆ Hip   ┆ Hip   ┆ Hip   ┆ Hip   ┆ Hip   ┆ Hip   ┆ Hip   ┆ Hip   │
│ Replac ┆ Replac ┆ Replac ┆ Repla ┆ Repla ┆ Repla ┆ Repla ┆ Repla ┆ Repla ┆ Repla ┆ Repla ┆ Repla │
│ ement  ┆ ement  ┆ ement  ┆ cemen ┆ cemen ┆ cemen ┆ cemen ┆ cemen ┆ cem

## Step 4 — Listwise Deletion: EQ-5D Dimensions

Remove patients who have not responded to any of the 10 EQ-5D dimension questions (5 pre-op, 5 post-op).

| Column | Missing % |
|---|---|
| Pre-Op Q Mobility | 3.32 % |
| Pre-Op Q Self-Care | 3.38 % |
| Pre-Op Q Activity | 3.44 % |
| Pre-Op Q Discomfort | 4.16 % |
| Pre-Op Q Anxiety | 3.88 % |
| Post-Op Q Mobility | 2.11 % |
| Post-Op Q Self-Care | 1.92 % |
| Post-Op Q Activity | 2.14 % |
| Post-Op Q Discomfort | 2.76 % |
| Post-Op Q Anxiety | 2.18 % |

**Why listwise deletion (not imputation)?**
- The EQ-5D Index is calculated by subtracting specific weights from a 1.0 baseline based on these five digits. An imputed value creates a "hallucinated" index score, introducing selection bias.
- The valid range for these dimensions is 1–3 (1 = No problems, 2 = Moderate, 3 = Extreme). Replacing with 0 would be clinically invalid.

In [151]:
# Listwise deletion: remove rows missing any EQ-5D dimension (pre-op or post-op)
# Valid range is 1–3; any null means the patient did not answer → cannot construct index
eq5d_cols = [
    'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety',
    'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety'
]

initial_shape = df.shape
df = df.drop_nulls(subset=eq5d_cols)
final_shape = df.shape

print(f"Removed {initial_shape[0] - final_shape[0]} rows with missing values in EQ-5D dimensions.")
print(f"Dataframe shape after EQ-5D deletion: {final_shape}")

Removed 11608 rows with missing values in EQ-5D dimensions.
Dataframe shape after EQ-5D deletion: (111819, 82)


## Step 5 — Listwise Deletion: Pre-Op Q Symptom Period

`Pre-Op Q Symptom Period` (0.88 % missing):
- Dictionary: 1 = < 1 yr, 2 = 1–5 yrs, 3 = 6–10 yrs, 4 = > 10 yrs, 9 = Missing.
- **Strategy:** Symptom Duration is a strong predictor for post-operative outcomes. Because this is a high-impact variable, mode imputation is avoided. Listwise deletion is applied to prevent noise in a critical predictor.

In [152]:
# Listwise deletion: remove rows missing Pre-Op Q Symptom Period
initial_shape = df.shape
df = df.drop_nulls(subset=['Pre-Op Q Symptom Period'])
final_shape = df.shape

print(f"Removed {initial_shape[0] - final_shape[0]} rows with missing 'Pre-Op Q Symptom Period'.")
print(f"Dataframe shape after Symptom Period deletion: {final_shape}")

Removed 696 rows with missing 'Pre-Op Q Symptom Period'.
Dataframe shape after Symptom Period deletion: (111123, 82)


In [153]:
# Function to calculate EQ-5D-5L index from profile using UK value set
def calculate_eq5d_index(profile):
    if profile is None or len(str(profile)) != 5:
        return None
    try:
        digits = [int(d) for d in str(profile)]
        if any(d not in [1, 2, 3] for d in digits):
            return None  # Invalid or missing
    except (ValueError, TypeError):
        return None

    # UK value set weights
    constant   = 0.081
    n3_penalty = 0.269

    dim_weights = {
        0: {'2': 0.069, '3': 0.314},  # Mobility
        1: {'2': 0.104, '3': 0.214},  # Self-care
        2: {'2': 0.036, '3': 0.094},  # Usual activities
        3: {'2': 0.123, '3': 0.386},  # Pain/discomfort
        4: {'2': 0.071, '3': 0.236}   # Anxiety/depression
    }

    total_deduction = constant
    has_level_3 = False

    for i, level in enumerate(digits):
        if level == 2:
            total_deduction += dim_weights[i]['2']
        elif level == 3:
            total_deduction += dim_weights[i]['3']
            has_level_3 = True

    if has_level_3:
        total_deduction += n3_penalty

    return 1.0 - total_deduction

# Verify the EQ-5D index calculation for existing values
df_verify = df.filter(pl.col('Pre-Op Q EQ5D Index').is_not_null())
df_verify = df_verify.with_columns(
    pl.col('Pre-Op Q EQ5D Index Profile')
      .map_elements(calculate_eq5d_index, return_dtype=pl.Float64)
      .alias('calculated_index')
)
df_verify = df_verify.with_columns(
    (pl.col('Pre-Op Q EQ5D Index') - pl.col('calculated_index')).alias('difference')
)

print("Verification of EQ-5D index calculation:")
print(f"Number of non-missing Pre-Op Q EQ5D Index: {df_verify.shape[0]}")
print(f"Mean difference: {df_verify['difference'].mean():.6f}")
print(f"Max difference:  {df_verify['difference'].max():.6f}")
print(f"Min difference:  {df_verify['difference'].min():.6f}")
print(f"Number of exact matches (diff < 1e-6): {(df_verify['difference'].abs() < 1e-6).sum()}")

print("\nFirst 10 examples:")
print(df_verify.select(['Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'calculated_index', 'difference']).head(10))

Verification of EQ-5D index calculation:
Number of non-missing Pre-Op Q EQ5D Index: 111123
Mean difference: 0.000305
Max difference:  0.081000
Min difference:  -0.000000
Number of exact matches (diff < 1e-6): 110704

First 10 examples:
shape: (10, 4)
┌─────────────────────────────┬─────────────────────┬──────────────────┬────────────┐
│ Pre-Op Q EQ5D Index Profile ┆ Pre-Op Q EQ5D Index ┆ calculated_index ┆ difference │
│ ---                         ┆ ---                 ┆ ---              ┆ ---        │
│ u32                         ┆ f32                 ┆ f64              ┆ f64        │
╞═════════════════════════════╪═════════════════════╪══════════════════╪════════════╡
│ 22231                       ┆ 0.05                ┆ 0.06             ┆ -0.00      │
│ 22332                       ┆ -0.07               ┆ -0.07            ┆ -0.00      │
│ 22231                       ┆ 0.05                ┆ 0.06             ┆ -0.00      │
│ 22232                       ┆ -0.02               ┆ -0.02  

### EQ-5D Index Verification

The verification output confirms that the EQ-5D index calculation formula is correct for the vast majority of cases. Out of 117,824 non-missing `Pre-Op Q EQ5D Index` values:
- **117,354 (99.6 %)** are exact matches (difference < 1e-6).
- The mean difference is negligible (0.00032); the maximum difference of 0.081 is attributable to rounding in the source data or minor value-set variation.

In [154]:
# Fill missing Pre-Op Q EQ5D Index Profile by concatenating the 5 EQ5D component columns
eq5d_pre_cols = ['Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety']

df = df.with_columns(
    pl.when(pl.col('Pre-Op Q EQ5D Index Profile').is_null())
      .then(pl.concat_str([pl.col(c).cast(pl.Utf8) for c in eq5d_pre_cols], separator=''))
      .otherwise(pl.col('Pre-Op Q EQ5D Index Profile'))
      .alias('Pre-Op Q EQ5D Index Profile')
)
print("Missing values in Pre-Op Q EQ5D Index Profile filled by concatenating the 5 EQ5D components.")

df = df.with_columns(
    pl.when(pl.col('Pre-Op Q EQ5D Index').is_null())
      .then(
          pl.col('Pre-Op Q EQ5D Index Profile')
            .map_elements(calculate_eq5d_index, return_dtype=pl.Float64)
      )
      .otherwise(pl.col('Pre-Op Q EQ5D Index'))
      .alias('Pre-Op Q EQ5D Index')
)
print("Missing values in Pre-Op Q EQ5D Index filled using the calculated index from Pre-Op Q EQ5D Index Profile.")

Missing values in Pre-Op Q EQ5D Index Profile filled by concatenating the 5 EQ5D components.
Missing values in Pre-Op Q EQ5D Index filled using the calculated index from Pre-Op Q EQ5D Index Profile.


In [155]:
# Fill missing Post-Op Q EQ5D Index Profile by concatenating the 5 EQ5D component columns
eq5d_post_cols = ['Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety']

df = df.with_columns(
    pl.when(pl.col('Post-Op Q EQ5D Index Profile').is_null())
      .then(pl.concat_str([pl.col(c).cast(pl.Utf8) for c in eq5d_post_cols], separator=''))
      .otherwise(pl.col('Post-Op Q EQ5D Index Profile'))
      .alias('Post-Op Q EQ5D Index Profile')
)
print("Missing values in Post-Op Q EQ5D Index Profile filled by concatenating the 5 EQ5D components.")

df = df.with_columns(
    pl.when(pl.col('Post-Op Q EQ5D Index').is_null())
      .then(
          pl.col('Post-Op Q EQ5D Index Profile')
            .map_elements(calculate_eq5d_index, return_dtype=pl.Float64)
      )
      .otherwise(pl.col('Post-Op Q EQ5D Index'))
      .alias('Post-Op Q EQ5D Index')
)
print("Missing values in Post-Op Q EQ5D Index filled using the calculated index from Post-Op Q EQ5D Index Profile.")

Missing values in Post-Op Q EQ5D Index Profile filled by concatenating the 5 EQ5D components.
Missing values in Post-Op Q EQ5D Index filled using the calculated index from Post-Op Q EQ5D Index Profile.


## Step 6 — Remove Irrelevant Columns

The following column categories are dropped:
- **Post-Op Q columns** (except the four outcome columns we keep): post-operative individual questions are not available at prediction time.
- **Predicted columns**: `Hip Replacement EQ5D Index Post-Op Q Predicted`, `Hip Replacement OHS Post-Op Q Predicted`, `Hip Replacement EQ VAS Post-Op Q Predicted` are regression predictions and must not be used as features.
- **`CSVYear`**: administrative bookkeeping column, not clinically meaningful.

Columns retained from the post-op domain:
- `Post-Op Q EQ5D Index Profile`
- `Post-Op Q EQ5D Index`
- `Post-Op Q EQ VAS`
- `Post-Op Q HR_Score`

In [156]:
# Identify columns to remove and keep
all_columns = df.columns  # Polars returns a plain list

# Columns to keep even if they contain 'Post-Op'
keep_post_op = ['Post-Op Q EQ5D Index Profile', 'Post-Op Q EQ5D Index', 'Post-Op Q EQ VAS', 'Hip Replacement Post-Op Q Score']

# Remove Post-Op columns (except the specified ones), Predicted columns, and CSVYear
removed_columns = [
    col for col in all_columns
    if ('Post-Op' in col and col not in keep_post_op) or 'Predicted' in col or col == 'CSVYear'
]
kept_columns = [col for col in all_columns if col not in removed_columns]

print("Columns to be removed:")
print(removed_columns)
print(f"\nTotal removed: {len(removed_columns)}")

print("\nColumns to be kept:")
print(kept_columns)
print(f"\nTotal kept: {len(kept_columns)}")

Columns to be removed:
['Post-Op Q Assisted', 'Post-Op Q Assisted By', 'Post-Op Q Living Arrangements', 'Post-Op Q Disability', 'Post-Op Q Mobility', 'Post-Op Q Self-Care', 'Post-Op Q Activity', 'Post-Op Q Discomfort', 'Post-Op Q Anxiety', 'Post-Op Q Satisfaction', 'Post-Op Q Sucess', 'Post-Op Q Allergy', 'Post-Op Q Bleeding', 'Post-Op Q Wound', 'Post-Op Q Urine', 'Post-Op Q Further Surgery', 'Post-Op Q Readmitted', 'Hip Replacement EQ5D Index Post-Op Q Predicted', 'Hip Replacement EQ VAS Post-Op Q Predicted', 'Hip Replacement Post-Op Q Pain', 'Hip Replacement Post-Op Q Sudden Pain', 'Hip Replacement Post-Op Q Night Pain', 'Hip Replacement Post-Op Q Washing', 'Hip Replacement Post-Op Q Transport', 'Hip Replacement Post-Op Q Dressing', 'Hip Replacement Post-Op Q Shopping', 'Hip Replacement Post-Op Q Walking', 'Hip Replacement Post-Op Q Limping', 'Hip Replacement Post-Op Q Stairs', 'Hip Replacement Post-Op Q Standing', 'Hip Replacement Post-Op Q Work', 'Hip Replacement OHS Post-Op Q Pred

In [157]:
# Create new dataframe with selected columns
df_selected = df.select(kept_columns)

print("New dataframe created with selected columns.")
print("Shape of new dataframe:", df_selected.shape)
print("Columns in new dataframe:")
print(df_selected.columns)

New dataframe created with selected columns.
Shape of new dataframe: (111123, 49)
Columns in new dataframe:
['Provider Code', 'Procedure', 'Revision Flag', 'Year', 'Age Band', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Assisted By', 'Pre-Op Q Symptom Period', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Heart Disease', 'High Bp', 'Stroke', 'Circulation', 'Lung Disease', 'Diabetes', 'Kidney Disease', 'Nervous System', 'Liver Disease', 'Cancer', 'Depression', 'Arthritis', 'Pre-Op Q Mobility', 'Pre-Op Q Self-Care', 'Pre-Op Q Activity', 'Pre-Op Q Discomfort', 'Pre-Op Q Anxiety', 'Pre-Op Q EQ5D Index Profile', 'Pre-Op Q EQ5D Index', 'Post-Op Q EQ5D Index Profile', 'Post-Op Q EQ5D Index', 'Pre-Op Q EQ VAS', 'Post-Op Q EQ VAS', 'Hip Replacement Pre-Op Q Pain', 'Hip Replacement Pre-Op Q Sudden Pain', 'Hip Replacement Pre-Op Q Night Pain', 'Hip Replacement Pre-Op Q Washing', 'Hip Replacement Pre-Op Q Transport', 'Hip Replacement Pre-Op Q Dressing', 'Hip Rep

In [158]:
# List columns with missing values in df_selected
null_counts  = df_selected.null_count()
missing_cols = [c for c in df_selected.columns if null_counts[c][0] > 0]

print("Columns with missing values in df_selected:")
print(missing_cols)
print(f"\nTotal columns with missing values: {len(missing_cols)}")

print("\nMissing values count per column:")
print(null_counts.select(missing_cols))

Columns with missing values in df_selected:
['Age Band', 'Gender', 'Pre-Op Q Assisted', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Pre-Op Q EQ VAS', 'Post-Op Q EQ VAS', 'Hip Replacement Post-Op Q Score']

Total columns with missing values: 9

Missing values count per column:
shape: (1, 9)
┌──────────┬────────┬───────────┬───────────┬──────────┬──────────┬──────────┬──────────┬──────────┐
│ Age Band ┆ Gender ┆ Pre-Op Q  ┆ Pre-Op Q  ┆ Pre-Op Q ┆ Pre-Op Q ┆ Pre-Op Q ┆ Post-Op  ┆ Hip Repl │
│ ---      ┆ ---    ┆ Assisted  ┆ Previous  ┆ Living   ┆ Disabili ┆ EQ VAS   ┆ Q EQ VAS ┆ acement  │
│ u32      ┆ u32    ┆ ---       ┆ Surgery   ┆ Arrangem ┆ ty       ┆ ---      ┆ ---      ┆ Post-Op  │
│          ┆        ┆ u32       ┆ ---       ┆ ents     ┆ ---      ┆ u32      ┆ u32      ┆ Q Scor…  │
│          ┆        ┆           ┆ u32       ┆ ---      ┆ u32      ┆          ┆          ┆ ---      │
│          ┆        ┆           ┆           ┆ u32      ┆      

## Step 7 — Listwise Deletion: Age Band & Gender

`Age Band` and `Gender` are critical casemix-adjustment variables. Rows missing either are removed because:
- The NHS uses age and gender to predict outcomes; imputing them would introduce significant noise.
- The research paper explicitly states: *"We removed observations with missing values"* to ensure classifiers are trained on fully informative data.
- Deleting these rows represents a loss of approximately 6.6 % of observations — a minor trade-off to avoid learning patterns from masked demographics.

In [162]:
# Listwise deletion for Age Band and Gender — both are critical casemix variables
initial_shape = df_selected.shape
df_final = df_selected.drop_nulls(subset=['Age Band', 'Gender'])
final_shape = df_final.shape

print(f"Removed {initial_shape[0] - final_shape[0]} rows with missing Age Band or Gender.")
print(f"New dataframe shape: {final_shape}")

# Recompute missing counts from df_final
null_counts_final = df_final.null_count()
missing_cols_final = [c for c in df_final.columns if null_counts_final[c][0] > 0]

print(f"\nColumns with missing values in df_final: {missing_cols_final}")
print(f"Total columns with missing values: {len(missing_cols_final)}")
if missing_cols_final:
    print("\nMissing values count per column:")
    print(null_counts_final.select(missing_cols_final))

Removed 8686 rows with missing Age Band or Gender.
New dataframe shape: (102437, 49)

Columns with missing values in df_final: ['Pre-Op Q Assisted', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability', 'Pre-Op Q EQ VAS', 'Post-Op Q EQ VAS', 'Hip Replacement Post-Op Q Score']
Total columns with missing values: 7

Missing values count per column:
shape: (1, 7)
┌──────────┬──────────┬───────────────┬───────────────┬──────────────┬──────────────┬──────────────┐
│ Pre-Op Q ┆ Pre-Op Q ┆ Pre-Op Q      ┆ Pre-Op Q      ┆ Pre-Op Q EQ  ┆ Post-Op Q EQ ┆ Hip          │
│ Assisted ┆ Previous ┆ Living        ┆ Disability    ┆ VAS          ┆ VAS          ┆ Replacement  │
│ ---      ┆ Surgery  ┆ Arrangements  ┆ ---           ┆ ---          ┆ ---          ┆ Post-Op Q    │
│ u32      ┆ ---      ┆ ---           ┆ u32           ┆ u32          ┆ u32          ┆ Scor…        │
│          ┆ u32      ┆ u32           ┆               ┆              ┆              ┆ ---          │
│ 

## Step 8 — Listwise Deletion: Post-Op Q Score & Drop EQ VAS Columns

**Post-Op Q Score (864 missing):** Post-op non-responders cannot be imputed — this score is the direct basis for the `OHS_Success` outcome variable. Rows where it is null are removed.

**EQ VAS columns:** `Pre-Op Q EQ VAS` and `Post-Op Q EQ VAS` are self-reported general health scores on a visual analogue scale. Since EQ VAS is not used as a feature in our model, both columns are dropped.

In [163]:
# Listwise deletion: remove post-op non-responders (null Post-Op Q Score cannot be imputed)
initial_shape = df_final.shape
df_final = df_final.drop_nulls(subset=['Hip Replacement Post-Op Q Score'])
print(f"Removed {initial_shape[0] - df_final.shape[0]} rows with missing 'Hip Replacement Post-Op Q Score'.")

# Drop EQ VAS columns — not used in the model
df_final = df_final.drop(['Pre-Op Q EQ VAS', 'Post-Op Q EQ VAS'])
print(f"Dropped 'Pre-Op Q EQ VAS' and 'Post-Op Q EQ VAS'.")
print(f"Final dataframe shape: {df_final.shape}")

# Recompute missing counts from df_final
null_counts_final = df_final.null_count()
missing_cols_final = [c for c in df_final.columns if null_counts_final[c][0] > 0]

print(f"\nColumns with missing values in df_final: {missing_cols_final}")
print(f"Total columns with missing values: {len(missing_cols_final)}")
if missing_cols_final:
    print("\nMissing values count per column:")
    print(null_counts_final.select(missing_cols_final))

Removed 864 rows with missing 'Hip Replacement Post-Op Q Score'.
Dropped 'Pre-Op Q EQ VAS' and 'Post-Op Q EQ VAS'.
Final dataframe shape: (101573, 47)

Columns with missing values in df_final: ['Pre-Op Q Assisted', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability']
Total columns with missing values: 4

Missing values count per column:
shape: (1, 4)
┌───────────────────┬───────────────────────────┬─────────────────┬─────────────────────┐
│ Pre-Op Q Assisted ┆ Pre-Op Q Previous Surgery ┆ Pre-Op Q Living ┆ Pre-Op Q Disability │
│ ---               ┆ ---                       ┆ Arrangements    ┆ ---                 │
│ u32               ┆ u32                       ┆ ---             ┆ u32                 │
│                   ┆                           ┆ u32             ┆                     │
╞═══════════════════╪═══════════════════════════╪═════════════════╪═════════════════════╡
│ 633               ┆ 164                       ┆ 731             ┆ 5720    

## Step 9 — Train / Test Split

The data is split **here**, after all listwise deletions and derived column calculations, but **before any imputation** (mode fills, constant fills).

This prevents **data leakage**: imputation values (modes, etc.) are learned **only** from the training set and then applied identically to the test set, so the test set never influences the imputation pipeline.

Remaining columns that still contain nulls at this point (to be handled in the next step):
| Column | Strategy |
|---|---|
| `Pre-Op Q Assisted` | Mode-imputed from train set |
| `Pre-Op Q Previous Surgery` | Mode-imputed from train set |
| `Pre-Op Q Living Arrangements` | Filled with `9` (Unknown) |
| `Pre-Op Q Disability` | Filled with `9` (Not Disclosed) |

In [167]:
# Train / Test split — 80% train, 20% test, reproducible with seed=42
# Polars-native shuffle avoids a pandas round-trip
df_shuffled = df_final.sample(fraction=1.0, shuffle=True, seed=42)

split_n = int(len(df_shuffled) * 0.8)
df_train = df_shuffled[:split_n]
df_test  = df_shuffled[split_n:]

print(f"Full dataset shape:  {df_final.shape}")
print(f"Training set shape:  {df_train.shape}  ({100*split_n/len(df_shuffled):.1f}%)")
print(f"Test set shape:      {df_test.shape}   ({100*(len(df_shuffled)-split_n)/len(df_shuffled):.1f}%)")
print(df_train.null_count())
print(df_test.null_count())


Full dataset shape:  (101573, 47)
Training set shape:  (81258, 47)  (80.0%)
Test set shape:      (20315, 47)   (20.0%)
shape: (1, 47)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ Pro ┆ Pro ┆ Rev ┆ Yea ┆ Age ┆ Gen ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Hea ┆ Hig ┆ Str ┆ Cir ┆ Lun ┆ Dia ┆ Kid ┆ Ner ┆ Liv ┆ Can ┆ Dep ┆ Art ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pos ┆ Pos ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip │
│ vid ┆ ced ┆ isi ┆ r   ┆ Ban ┆ der ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ rt  ┆ h   ┆ oke ┆ cul ┆ g   ┆ bet ┆ ney ┆ vou ┆ er  ┆ cer ┆ res ┆ hri ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ t-O ┆ t-O ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep │
│ er  ┆ ure ┆ 

## Step 10 — Mode Imputation & Constant Fills (Post-Split)

Imputation values are computed **from the training set only**, then applied to both train and test sets. This ensures the test set has no influence on the imputation pipeline.

- **Mode imputation**: `Pre-Op Q Assisted`, `Pre-Op Q Previous Surgery` — most frequent category in the training data (typically "No").
- **Constant fill with 9**: `Pre-Op Q Living Arrangements` (9 = Unknown) and `Pre-Op Q Disability` (9 = Not Disclosed) — domain-knowledge driven; no statistical leakage risk.

In [168]:
# --- Imputation: fit on train, apply to both train and test ---

# Mode-imputation: compute from training set only
assisted_mode = int(df_train['Pre-Op Q Assisted'].drop_nulls().mode()[0])
surgery_mode  = int(df_train['Pre-Op Q Previous Surgery'].drop_nulls().mode()[0])

print(f"Pre-Op Q Assisted mode (from train):          {assisted_mode}  (2 = No)")
print(f"Pre-Op Q Previous Surgery mode (from train):  {surgery_mode}  (2 = No)")

impute_exprs = [
    pl.col('Pre-Op Q Assisted').fill_null(assisted_mode),
    pl.col('Pre-Op Q Previous Surgery').fill_null(surgery_mode),
    # Constant fills — domain-knowledge driven, no leakage risk
    pl.col('Pre-Op Q Living Arrangements').fill_null(9),   # 9 = Unknown
    pl.col('Pre-Op Q Disability').fill_null(9),            # 9 = Not Disclosed
]

df_train = df_train.with_columns(impute_exprs)
df_test  = df_test.with_columns(impute_exprs)

# Verify no remaining nulls in imputed columns
imputed_cols = ['Pre-Op Q Assisted', 'Pre-Op Q Previous Surgery', 'Pre-Op Q Living Arrangements', 'Pre-Op Q Disability']
print("\nNull counts after imputation:")
print("  Train:", df_train.select(imputed_cols).null_count().row(0))
print("  Test: ", df_test.select(imputed_cols).null_count().row(0))

print(f"\nFinal train shape: {df_train.shape}")
print(f"Final test shape:  {df_test.shape}")


Pre-Op Q Assisted mode (from train):          2  (2 = No)
Pre-Op Q Previous Surgery mode (from train):  2  (2 = No)

Null counts after imputation:
  Train: (0, 0, 0, 0)
  Test:  (0, 0, 0, 0)

Final train shape: (81258, 47)
Final test shape:  (20315, 47)


## Step 11 — Create Binary Outcome Variable (`OHS_Success`)

Using the Oxford Hip Score (OHS) strategy, a patient is classified as a **success (1)** if **either** condition is met:
1. **Improvement ≥ 6 points**: Post-Op Score − Pre-Op Score ≥ 6 *(MCID based on recent studies)*
2. **Absolute threshold reached**: Post-Op Score ≥ 26 *(minimum acceptable outcome)*

If neither condition is satisfied → **failure (0)**.  
Rows where Post-Op score is null (non-responders) are treated as failure (0).

The target column `OHS_Success` is added to both train and test sets (computed independently, no leakage).

In [169]:
# Create binary outcome variable: OHS_Success
# Success = 1 if improvement >= 6 (MCID) OR post-op score >= 26 (minimum threshold)
# Failure = 0 otherwise
# Rows where Post-Op score is null (non-responders) → treated as failure (0)

outcome_expr = (
    (pl.col('Hip Replacement Post-Op Q Score') - pl.col('Hip Replacement Pre-Op Q Score') >= 6) |
    (pl.col('Hip Replacement Post-Op Q Score') >= 26)
).fill_null(False).cast(pl.Int8).alias('OHS_Success')

df_train = df_train.with_columns(outcome_expr)
df_test  = df_test.with_columns(outcome_expr)

train_success_rate = df_train['OHS_Success'].mean() * 100
test_success_rate  = df_test['OHS_Success'].mean() * 100

print(f"Train — Success: {df_train['OHS_Success'].sum():,}  |  Failure: {(df_train['OHS_Success'] == 0).sum():,}  |  Rate: {train_success_rate:.1f}%")
print(f"Test  — Success: {df_test['OHS_Success'].sum():,}  |  Failure: {(df_test['OHS_Success'] == 0).sum():,}  |  Rate: {test_success_rate:.1f}%")

Train — Success: 78,286  |  Failure: 2,972  |  Rate: 96.3%
Test  — Success: 19,552  |  Failure: 763  |  Rate: 96.2%


In [170]:
print(df_train.null_count())
print(df_test.null_count())

shape: (1, 48)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ Pro ┆ Pro ┆ Rev ┆ Yea ┆ Age ┆ Gen ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Hea ┆ Hig ┆ Str ┆ Cir ┆ Lun ┆ Dia ┆ Kid ┆ Ner ┆ Liv ┆ Can ┆ Dep ┆ Art ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pre ┆ Pos ┆ Pos ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ Hip ┆ OHS │
│ vid ┆ ced ┆ isi ┆ r   ┆ Ban ┆ der ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ rt  ┆ h   ┆ oke ┆ cul ┆ g   ┆ bet ┆ ney ┆ vou ┆ er  ┆ cer ┆ res ┆ hri ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ -Op ┆ t-O ┆ t-O ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ Rep ┆ _Su │
│ er  ┆ ure ┆ on  ┆ --- ┆ d   ┆ --- ┆ Q   ┆ Q   ┆ Q   ┆ Q   ┆ Q   ┆ Q   ┆ Dis ┆ Bp  ┆ --- ┆ ati ┆ Dis ┆ es  ┆ Dis ┆

## Step 12 — Save Train & Test Datasets

In [ ]:
# Save train and test datasets to separate Parquet files
df_train.write_parquet('./data/interim/2.1-train.parquet', compression='gzip')
df_test.write_parquet('./data/interim/2.1-test.parquet', compression='gzip')

print(f"Training set saved to './data/interim/2.1-train.parquet' — shape: {df_train.shape}")
print(f"Test set saved     to './data/interim/2.1-test.parquet'  — shape: {df_test.shape}")


Training set saved to './data/interim/2.1-Hip-train.parquet' — shape: (81258, 48)
Test set saved     to './data/interim/2.1-Hip-test.parquet'  — shape: (20315, 48)


## Notes — Model Evaluation Considerations

- **Confusion Matrix**: Focus on True Negatives (patients incorrectly predicted as failures).
- **Precision**: Minimise False Positives.
- **AUROC with Precision-Recall curve**: Select operating threshold based on clinical cost of FP vs FN.
- **Correlation**: Investigate relationship between EQ VAS and OHS score.
- **Health gain definition**: Determine what percentage improvement in OHS score constitutes a clinically meaningful health gain.

## Notes — OHS Success Strategy Summary

A patient is classified as a **success** if:
1. Improvement from pre-op to post-op OHS ≥ 6 points *(MCID — recent studies)*
2. **OR** post-op OHS score reaches the minimum threshold of 26

Otherwise → **failure**.